# Memory names: is the intraday (open/early) move indicative of the CLOSE? — token-free (Colab)

MEMORY/STORAGE only: MU, WDC, STX, SNDK. Question: when these are green early (like today), does it HOLD
to the close (gap-and-go) or FADE (gap-and-fade)? Answered two ways:
1. **Daily OHLC over years** (robust): does the OPEN gap predict the OPEN→CLOSE move? On up-gap days, do
   they close above the open, and close green? corr(gap, intraday).
2. **Recent intraday** (30-min bars, ~60d): does the MORNING move persist into the afternoon/close?

Run top-to-bottom; last cell exports one file. yfinance (needs OHLC + intraday, so no Stooq fallback here).


## capture setup


In [ ]:
import builtins as _bi
REPORT_LINES=[]; _realprint=_bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np, yfinance as yf
except Exception:
    _pip('pandas'); _pip('numpy'); _pip('yfinance'); import pandas as pd, numpy as np, yfinance as yf

MEMORY = ['MU','WDC','STX','SNDK']   # DRAM/NAND/HDD storage complex
BIG_GAP = 0.01                        # 'big' up-gap threshold (1%)


## Load daily OHLC + build gap / intraday(open->close) / day returns


In [ ]:
def load_ohlc(t):
    df = yf.download(t, start='2005-01-01', progress=False, auto_adjust=True)
    if not len(df): return None
    o = df['Open']; c = df['Close']
    o = o.iloc[:,0] if hasattr(o,'columns') else o; c = c.iloc[:,0] if hasattr(c,'columns') else c
    x = pd.DataFrame({'open':np.asarray(o).ravel(), 'close':np.asarray(c).ravel()}, index=df.index).dropna()
    x['prev_close'] = x['close'].shift(1)
    x['gap'] = x['open']/x['prev_close'] - 1     # overnight gap
    x['oc']  = x['close']/x['open'] - 1           # intraday: open -> close
    x['day'] = x['close']/x['prev_close'] - 1
    return x.dropna()

data = {}
for t in MEMORY:
    try:
        x = load_ohlc(t)
        if x is not None and len(x)>50: data[t]=x; print(f'{t}: {len(x)} days {x.index[0].date()}->{x.index[-1].date()}')
        else: print(f'{t}: too little data, skipped')
    except Exception as e: print(f'{t}: load failed {e}')


## TEST 1 — daily: does an up-gap HOLD to the close, or FADE?


In [ ]:
def gap_stats(x, label):
    ug = x[x['gap']>0]
    bg = x[x['gap']>BIG_GAP]
    r  = x['gap'].corr(x['oc'])
    return (f'  {label:>6}: corr(gap,oc) {r:+.2f} | UP-GAP days n={len(ug):4d} mean intraday {1e4*ug["oc"].mean():+6.1f}bp '
            f'close>open {100*(ug["oc"]>0).mean():3.0f}% close green {100*(ug["day"]>0).mean():3.0f}% '
            f'|| BIG-gap(>1%) n={len(bg):4d} intraday {1e4*bg["oc"].mean():+6.1f}bp hold {100*(bg["oc"]>0).mean():3.0f}%')
print('KEY: corr(gap,oc) NEGATIVE = gap-and-FADE (morning green gives back); POSITIVE = gap-and-GO (holds/extends).')
print('     UP-GAP mean intraday NEGATIVE = the early pop tends to bleed into the close.\n')
allx=[]
for t in MEMORY:
    if t in data: print(gap_stats(data[t], t)); allx.append(data[t])
if allx:
    pool = pd.concat(allx); print(gap_stats(pool,'POOL'))


## TEST 2 — recent intraday (30-min bars, ~60d): does the MORNING move persist to the close?


In [ ]:
def intraday_persist(t):
    df = yf.download(t, period='60d', interval='30m', progress=False, auto_adjust=True)
    if not len(df): return None
    c = df['Close']; c = c.iloc[:,0] if hasattr(c,'columns') else c
    s = pd.Series(np.asarray(c).ravel(), index=df.index).dropna()
    try: s.index = s.index.tz_convert('US/Eastern')
    except Exception: pass
    rows=[]
    for day, grp in s.groupby(s.index.date):
        grp = grp.sort_index()
        if len(grp) < 6: continue
        op = grp.iloc[0]; mid = grp.iloc[len(grp)//2]; cl = grp.iloc[-1]
        rows.append((op, mid, cl))
    if len(rows)<10: return None
    R = pd.DataFrame(rows, columns=['op','mid','cl'])
    R['morning'] = R['mid']/R['op']-1; R['rest'] = R['cl']/R['mid']-1
    return R

pooled=[]
for t in MEMORY:
    try:
        R = intraday_persist(t)
        if R is None: print(f'{t}: no/low intraday data'); continue
        c = R['morning'].corr(R['rest'])
        upm = R[R['morning']>0]
        print(f'  {t}: days={len(R):3d} corr(morning,rest) {c:+.2f} | when UP by midday -> afternoon mean {1e4*upm["rest"].mean():+6.1f}bp, stays green {100*(upm["rest"]>0).mean():3.0f}%')
        pooled.append(R)
    except Exception as e: print(f'{t}: intraday failed {e}')
if pooled:
    P = pd.concat(pooled); upm=P[P['morning']>0]
    print(f'  POOL : days={len(P):3d} corr(morning,rest) {P["morning"].corr(P["rest"]):+.2f} | UP-by-midday -> afternoon {1e4*upm["rest"].mean():+6.1f}bp, stays green {100*(upm["rest"]>0).mean():3.0f}%')
print('\nNEGATIVE corr / negative afternoon after up-morning = the early move FADES into the close.')


## ⬇️ EXPORT — run LAST


In [ ]:
from datetime import datetime
fname='memory_intraday_report.txt'
hdr=['='*66,'MEMORY: intraday-indicative-of-close — export',
     f'names {MEMORY} | generated {datetime.now().strftime("%Y-%m-%d %H:%M")}','='*66,'']
open(fname,'w').write('\n'.join(hdr)+'\n'+'\n'.join(REPORT_LINES)+'\n')
_realprint('wrote',fname,'—',len(REPORT_LINES),'lines')
if not REPORT_LINES: _realprint('!! empty — run cells above first')
try:
    from google.colab import files; files.download(fname); _realprint('download started')
except Exception as e:
    _realprint('not in Colab / blocked:',e,'- grab it from the folder icon (left)')
